In [ ]:
from molearn.models.ldm import MiniLDM
from molearn.models.foldingnet import AutoEncoder
from molearn.data import PDBData

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import os
from collections import deque


/home/alexandros/anaconda3/envs/molearn/lib/python3.11/site-packages/molearn/loss_functions/torch_protein_energy_utils.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
checkpoint_file ="/home/alexandros/root/phd/dev/molearn/examples/foldingnet_checkpoints/checkpoint_no_optimizer_state_dict_epoch167_loss0.003259085263643.ckpt"
checkpoint = torch.load(checkpoint_file, map_location= torch.device('cpu'), weights_only=False)

model = AutoEncoder(**checkpoint['network_kwargs'])
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

training_set_files = [f'data{os.sep}MurD_open.pdb', f'data{os.sep}MurD_open.pdb']
test_set_file = f'data{os.sep}MurD_closed_apo.pdb'


print("Loading training set")
data = PDBData()
for f in training_set_files:
    data.import_pdb(f)
data.atomselect(atoms=['CA', 'C', 'CB', 'N', 'O'])
data.prepare_dataset()

In [ ]:
z_dim = 2
encoded = []

# for i in range(len(data.dataset)):
for i in range(10):
    with torch.no_grad():
        z = model.encode(data.dataset[i:i+1]).view(z_dim).cpu()
        encoded.append(z)

print("Done encoding")
encoded_dataset = encoded[:]

Done encoding


In [ ]:
encoded_data_tensor = torch.stack(encoded_dataset)
dataset = TensorDataset(encoded_data_tensor)

batch_size = 8
dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True
)

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
schedules = get_schedules(device)

loss_function = nn.MSELoss()
epochs = 1
optimizer = Adam
ldm = MiniLDM()

In [ ]:
train(ldm, dataloader, schedules, epochs, loss_function, optimizer, device)